In [11]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [4]:
from google.cloud import bigquery
import pandas as pd

# Inicializa el cliente de BigQuery
client = bigquery.Client(project='dataton-2024-team-01-cofares')

# Ejecuta la consulta y convierte los datos en un DataFrame de Pandas desde BigQuery datos_no_descriptions_eans
#query = "SELECT * FROM `dataton-2024-team-01-cofares.datos_cofares.datos_no_descriptions_eans`"
#df = client.query(query).to_dataframe()
#print(df.head())

# Ejecuta la consulta y convierte los datos en un DataFrame de Pandas
df = pd.read_csv("productos_no_description_df.csv")
df.head()

,codigo_web,nombre_completo_material,codigo_nacional,es_marca_propia,nombre_proveedor,nombre_matricula_nivel0,nombre_matricula_nivel1,nombre_matricula_nivel2,nombre_matricula_nivel3,nombre_matricula_nivel4,nombre_matricula_nivel5,txt_mas_informacion_del_producto,txt_instrucciones_de_uso,txt_composicion,ids_imagenes,URI_primera_imagen,eans
0,218790,ALLY PASTE 80 GR,2187900,False,BRAUN-MEDICAL.S.A.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],NaN,"[{'EAN13': '6926515481575', 'fecha_asignacion'..."
1,26083,VENOFL SECR AD CCL2 NORM T3,260834,False,THUASNE ESPAÑA. S.L.U.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],NaN,"[{'EAN13': '3111790260134', 'fecha_asignacion'..."
2,25749,FAJA DORSOLUMB SEMIR FX-213 T2,257490,False,ORLIMAN. S.L.U. (M.P.),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],NaN,"[{'EAN13': '8435025921245', 'fecha_asignacion'..."
3,186347,KUORA CREMA FACIAL EFECTO BOTOX 28 DIAS 1 ENVA...,1863478,False,ORTI PIERA. NURIA,CUIDADO PERSONAL (PEC),DERMOCOSMETICA FACIAL,ANTIEDAD,DIA,DIA,DIA,NaN,NaN,NaN,['186347.jpg'],gs://dataton-2024-team-01-cofares-datastore/im...,"[{'EAN13': '8470001863478', 'fecha_asignacion'..."
4,180556,ATASHI CELLULAR COSMETICS RITUAL ANTIEDAD DIA ...,1805560,False,PHERGAL. S.A.,CUIDADO PERSONAL (PEC),DERMOCOSMETICA FACIAL,ANTIEDAD,DIA,DIA,DIA,NaN,NaN,NaN,['180556.jpg'],gs://dataton-2024-team-01-cofares-datastore/im...,"[{'EAN13': '8429449052449', 'fecha_asignacion'..."


In [12]:
#EXTRAE SOLO 1 RESULTADO(el primero)
def google_custom_search_df(query, country):
    API_KEY = os.getenv('API_KEY')
    API_CUSTOM_SEARCH_ID = os.getenv('API_CUSTOM_SEARCH_ID')

    if not API_KEY or not API_CUSTOM_SEARCH_ID:
        print("Error: API Key o ID de búsqueda no están cargados correctamente")
        return pd.DataFrame()

    # URL request para obtener solo los primeros 10 resultados
    url = f"https://www.googleapis.com/customsearch/v1?key={API_KEY}&cx={API_CUSTOM_SEARCH_ID}&q={query}&start=1&gl={country}"
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Error en la petición: {response.status_code}")
        return pd.DataFrame()

    data = response.json()

# Extraer solo el primer resultado
    items = data.get("items", [])
    if items:
        items = items[:1]  # Solo tomar el primer resultado

    if not items:
        print("No se encontraron resultados")
        return pd.DataFrame()

    # Crear un DataFrame con los resultados de la búsqueda
    df = pd.DataFrame(items, columns=['title', 'link']) 

    return df


In [6]:
#-------------------SCRAPING del texto producto

#Extraer el texto de los articulos con la libreria newspapper y, en su defecto, con beautiful soup
from bs4 import BeautifulSoup
from newspaper import Article

def scrape_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        if not article.text:
            # Si Newspaper no pudo obtener el texto, intenta con BeautifulSoup
            page = requests.get(url)
            soup = BeautifulSoup(page.content, 'html.parser')
            
            # Busca contenido en etiquetas <p> o <div>
            article_text = ' '.join([p.get_text() for p in soup.find_all(['p', 'div'])])
            
            return article_text
        return article.text
    except Exception as e:
        print(f"Error al scrapear el artículo: {str(e)}")
        return ""

def scrape_articles_in_dataframe(df):
    scraped_texts = []  # Aquí almacenaremos el texto de los artículos

# Solo procesar el primer enlace
    if not df.empty:
        url = df['link'].iloc[0]  # Obtener solo el primer enlace
        scraped_text = scrape_article(url)
        scraped_texts.append(scraped_text)

    # Agregamos los textos como una nueva columna en el DataFrame
    df['scraped_text'] = scraped_texts

    return scraped_texts


In [7]:
def extract_image_url(url):
    try:
        # Solicitar el contenido de la página
        page = requests.get(url)
        soup = BeautifulSoup(page.content, 'html.parser')
        
        # Buscar la etiqueta img relevante
        img_tag = soup.find('img', {'class': 'img-fluid'})  # Ajusta según la clase o atributo de la imagen
        
        if img_tag and 'src' in img_tag.attrs:
            img_url = img_tag['src']
            return img_url
        else:
            return "No se encontró la URL de la imagen."
    except Exception as e:
        return f"Error al extraer la imagen: {str(e)}"

In [ ]:
def process_products(df):
    results = []

    # Limitar el DataFrame a los primeros 5 productos
    #df_limited = df.head(19000)

    for index, row in df.iterrows():
        query = row['codigo_nacional']
        country = "ES"
        
        # Buscar el producto
        df_productos = google_custom_search_df(query, country)
        
        if not df_productos.empty:
            url = df_productos['link'].iloc[0]
            
            # Scraping del texto y las URLs de imágenes
            scraped_text= scrape_article(url)
            image_urls = extract_image_url(url)
            # Almacenar resultados
            results.append({
                'codigo_nacional': query,
                'scraped_text': scraped_text,
                'image_urls': image_urls  # Almacenar también las URLs de imágenes
            })
        else:
            print(f"No se encontraron resultados para el producto con código: {query}")

    # Convertir resultados a DataFrame
    return pd.DataFrame(results)

# Llamar a la función para procesar los productos
df_resultados = process_products(df)
print(df_resultados)

In [7]:
def scrape_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        if not article.text:
            # Si Newspaper no pudo obtener el texto, intenta con BeautifulSoup
            page = requests.get(url)
            soup = BeautifulSoup(page.content, 'html.parser')
            
            # Busca contenido en etiquetas <p> o <div>
            article_text = ' '.join([p.get_text() for p in soup.find_all(['p', 'div'])])
            
            # Extraer URLs de imágenes
            image_urls = [img['src'] for img in soup.find_all('img') if 'src' in img.attrs]
            
            return article_text, image_urls
        return article.text, []
    except Exception as e:
        print(f"Error al scrapear el artículo: {str(e)}")
        return "", []

def scrape_articles_in_dataframe(df):
    scraped_texts = []  # Aquí almacenaremos el texto de los artículos
    image_urls_list = []  # Aquí almacenaremos las URLs de las imágenes

    # Solo procesar el primer enlace
    if not df.empty:
        url = df['link'].iloc[0]  # Obtener solo el primer enlace
        scraped_text, image_urls = scrape_article(url)
        scraped_texts.append(scraped_text)
        image_urls_list.append(image_urls)

    # Agregamos los textos y las URLs de las imágenes como nuevas columnas en el DataFrame
    df['scraped_text'] = scraped_texts
    df['image_urls'] = image_urls_list

    return df

# Llama a la función scraping de los artículos
df = scrape_articles_in_dataframe(df_productos)
print(df)

                                               title  \
0  Comprar Paranix Loción Elimina Piojos Y Liendr...   

                                                link  \
0  https://www.welnia.com/piojos/paranix-locion-e...   

                                        scraped_text image_urls  
0  Descripción\n\nParanix Loción Elimina Piojos y...         []  


In [10]:
def extraer_ean(valor):
    # Si el valor es una cadena (string)
    if isinstance(valor, str):
        # Busca el EAN13 después de la comilla simple
        import re
        match = re.search(r"'EAN13': '(\d+)'", valor)
        if match:
            return match.group(1)
    
    # Si el valor es una lista de diccionarios
    elif isinstance(valor, list):
        if valor and isinstance(valor[0], dict):
            return valor[0].get('EAN13')
    
    return None

# Aplicar la función a la columna
df['ean_limpio'] = df['eans'].apply(extraer_ean)
print(df.head())

   codigo_web                           nombre_completo_material  \
0      218790                                   ALLY PASTE 80 GR   
1       26083                        VENOFL SECR AD CCL2 NORM T3   
2       25749                     FAJA DORSOLUMB SEMIR FX-213 T2   
3      186347  KUORA CREMA FACIAL EFECTO BOTOX 28 DIAS 1 ENVA...   
4      180556  ATASHI CELLULAR COSMETICS RITUAL ANTIEDAD DIA ...   

   codigo_nacional  es_marca_propia        nombre_proveedor  \
0          2187900            False      BRAUN-MEDICAL.S.A.   
1           260834            False  THUASNE ESPAÑA. S.L.U.   
2           257490            False  ORLIMAN. S.L.U. (M.P.)   
3          1863478            False       ORTI PIERA. NURIA   
4          1805560            False           PHERGAL. S.A.   

  nombre_matricula_nivel0 nombre_matricula_nivel1 nombre_matricula_nivel2  \
0                     NaN                     NaN                     NaN   
1                     NaN                     NaN         

In [6]:
import vertexai
from vertexai import generative_models as genai
from vertexai.generative_models import GenerationConfig

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")


# Model definition
multimodal_model = genai.GenerativeModel( "gemini-1.5-flash",
generation_config= GenerationConfig(temperature=0))

In [5]:
def descripcion_producto_con_gemini(df):
    # Iterar sobre el DataFrame para procesar cada producto
    for index, row in df.iterrows():
        title = row['title']
        scraped_text = row['scraped_text']
        prompt = f"""
        Genera una descripción precisa del producto '{title}' basada en su texto scrapeado: {scraped_text}.
        Destaca las características clave del producto y asegúrate de que la descripción sea clara y concisa.
        Idioma: español. Codificación: UTF-8
        """
        try:
            response = multimodal_model.generate_content(prompt)
            descripcion = response.text
            print(f"Descripción de '{title}' generada con éxito.")
            return descripcion
        except Exception as e:
            print(f"Error al generar la descripción de '{title}' con Gemini: {str(e)}")
            return f"Error al generar la descripción de '{title}' con Gemini"

# Llamar a la función para generar la descripción
descripcion_producto = descripcion_producto_con_gemini(df)

# Guardar la descripción en un archivo CSV
import csv
with open("descripcion_producto.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([descripcion_producto])

print("\nDescripción del producto guardada en 'descripcion_producto.csv'")

Descripción de 'Comprar Iraltone DS Champú, 200 ml | Welnia' generada con éxito.

Descripción del producto guardada en 'descripcion_producto.csv'
